In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [2]:
crop_df=pd.read_csv("datasets/Crop_recommendation.csv")

In [3]:
crop_df.head()

,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


In [4]:
crop_df[crop_df.isnull().any(axis=1)]

,N,P,K,temperature,humidity,ph,rainfall,label


In [5]:
crop_df.duplicated().any()

np.False_

In [6]:
len(crop_df)

2200

In [7]:
crop_df=crop_df.drop_duplicates()

In [8]:
len(crop_df)

2200

In [9]:
numeric_cols = crop_df.select_dtypes(include=[np.number]).columns
outlier_summary = {}
for col in numeric_cols:
    Q1 = crop_df[col].quantile(0.25)
    Q3 = crop_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = crop_df[(crop_df[col] < lower_bound) | (crop_df[col] > upper_bound)][col]
    outlier_summary[col] = {
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": outliers.shape[0],
        "Outlier Values": outliers.values
    }
for col, info in outlier_summary.items():
    print("\nColumn:", col)
    print("Lower Bound:", info["Lower Bound"])
    print("Upper Bound:", info["Upper Bound"])
    print("Outliers:", info["Outlier Values"])
    print("Count:", info["Outlier Count"])


Column: N
Lower Bound: -73.875
Upper Bound: 179.125
Outliers: []
Count: 0

Column: P
Lower Bound: -32.0
Upper Bound: 128.0
Outliers: [130 144 131 140 134 130 145 139 141 138 144 136 136 145 132 133 140 132
 142 135 139 141 142 129 134 138 131 132 137 136 134 139 138 142 133 139
 134 140 139 136 139 133 130 135 140 132 132 142 140 133 135 145 136 129
 130 129 135 132 140 145 139 144 141 138 138 143 142 134 144 129 137 139
 144 139 133 143 140 137 144 143 140 144 141 144 143 137 144 143 141 142
 138 137 135 144 133 130 143 143 139 136 131 140 138 145 139 136 138 136
 134 143 145 141 136 136 141 129 138 137 132 139 143 144 143 135 130 142
 129 135 145 131 140 138 140 145 132 137 144 140]
Count: 138

Column: K
Lower Bound: -23.5
Upper Bound: 92.5
Outliers: [195 204 205 196 196 198 197 195 203 204 197 205 201 203 204 195 202 205
 197 204 201 197 198 201 199 205 203 197 200 197 203 203 198 202 196 201
 199 204 198 204 201 204 197 195 200 197 200 201 196 203 204 203 195 199
 202 195 202 195 

In [10]:
df_clean = crop_df.copy()

for col in numeric_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]


In [11]:
df_clean.head()

,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
7,94,53,40,20.277744,82.894086,5.718627,241.974195,rice
8,89,54,38,24.515881,83.535216,6.685346,230.446236,rice


In [12]:
df_clean['label'].unique()

array(['rice', 'maize', 'chickpea', 'kidneybeans', 'pigeonpeas',
       'mothbeans', 'mungbean', 'blackgram', 'lentil', 'pomegranate',
       'banana', 'mango', 'watermelon', 'muskmelon', 'orange', 'papaya',
       'coconut', 'cotton', 'jute', 'coffee'], dtype=object)

In [13]:
len(df_clean)

1846

In [14]:
encoder=LabelEncoder()

In [15]:
df_clean['Crops']=encoder.fit_transform(df_clean['label'])

In [16]:
df_clean.head()

,N,P,K,temperature,humidity,ph,rainfall,label,Crops
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice,18
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice,18
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice,18
7,94,53,40,20.277744,82.894086,5.718627,241.974195,rice,18
8,89,54,38,24.515881,83.535216,6.685346,230.446236,rice,18


In [17]:
df_clean=df_clean.drop(columns=['label'])

In [18]:
X=df_clean.drop(columns=['Crops'])
y=df_clean['Crops']

In [19]:
X.head()

,N,P,K,temperature,humidity,ph,rainfall
0,90,42,43,20.879744,82.002744,6.502985,202.935536
1,85,58,41,21.770462,80.319644,7.038096,226.655537
3,74,35,40,26.491096,80.158363,6.980401,242.864034
7,94,53,40,20.277744,82.894086,5.718627,241.974195
8,89,54,38,24.515881,83.535216,6.685346,230.446236


In [20]:
y

0       18
1       18
3       18
7       18
8       18
        ..
2195     4
2196     4
2197     4
2198     4
2199     4
Name: Crops, Length: 1846, dtype: int64

In [21]:
scaler=StandardScaler()

In [22]:
X_scaler=scaler.fit_transform(X)

In [23]:
X_scaler

array([[ 0.94386616, -0.14050707,  0.6173409 , ...,  0.54021372,
         0.0141572 ,  1.93559816],
       [ 0.81200377,  0.57059932,  0.49693187, ...,  0.46588231,
         0.80095938,  2.38811036],
       [ 0.5219065 , -0.45161612,  0.43672735, ...,  0.45875957,
         0.7161267 ,  2.69732378],
       ...,
       [ 1.68229557, -0.54050442, -0.16531782, ..., -0.11241614,
        -0.19224719,  1.37066955],
       [ 1.65592309, -0.58494857,  0.07550025, ..., -0.77918307,
         0.39028418,  0.49030165],
       [ 1.31308086, -1.20716667, -0.16531782, ..., -0.41399243,
         0.4212205 ,  0.75283786]], shape=(1846, 7))

In [24]:
X_train,X_test,y_train,y_test=train_test_split(X_scaler,y,random_state=32,test_size=0.25)

In [25]:
classfier=RandomForestClassifier()

In [26]:
classfier.fit(X_train,y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [27]:
classfier.score(X_train,y_train)

1.0

In [28]:
training_prediction=classfier.predict(X_train)
testing_prediction=classfier.predict(X_test)

In [29]:
Training_accuracy=accuracy_score(y_pred=training_prediction,y_true=y_train)
Testing_accuracy=accuracy_score(y_pred=testing_prediction,y_true=y_test)

In [30]:
print("training accuracy=",Training_accuracy," testing accuracy=",Testing_accuracy)

training accuracy= 1.0  testing accuracy= 0.9913419913419913


In [31]:
import pickle
with open("models/model.pkl","wb") as m:
    pickle.dump(classfier,m)
with open("models/encoder.pkl","wb") as e:
    pickle.dump(encoder,e)
with open("models/scaler.pkl","wb") as s:
    pickle.dump(scaler,s)